# L2L Bench Aviary Integration Demo

This notebook demonstrates the new aviary environment integration for L2L Bench.

The integration provides:
- `L2LEnvironment`: A gymnasium-style environment for scientific reasoning episodes
- `L2LTaskDataset`: Aviary TaskDataset for evaluation workflows
- `L2LState`: Pydantic state model tracking episode progress
- `Scorer` classes: LLM-as-judge scoring for hypothesis/interpretation quality

In [1]:
# imports
from pathlib import Path
import tempfile

from l2l_bench.aviary import (
    L2LEnvironment,
    L2LState,
    StepRecord,
    L2LTaskDataset,
    DummyScorer,
    LLMScorer,
)

print("All imports successful!")

All imports successful!


## 1. State Model Demo

The `L2LState` class tracks all episode state, following aviary's convention where tools can set `state.done` and `state.reward` directly.

In [2]:
# create a StepRecord to track a single experimental step
step = StepRecord(
    step_number=1,
    hypothesis="H1: MEK inhibitors will suppress MAPK signaling\nH2: No effect on MAPK",
    prediction="Prediction: Negative NES for MAPK pathway\nConfidence: high",
)

print("StepRecord created:")
print(f"  Step number: {step.step_number}")
print(f"  Hypothesis: {step.hypothesis[:50]}...")
print(f"  Experiment run: {step.experiment is not None}")
print(f"  Interpretation submitted: {step.interpretation is not None}")

StepRecord created:
  Step number: 1
  Hypothesis: H1: MEK inhibitors will suppress MAPK signaling
H2...
  Experiment run: False
  Interpretation submitted: False


In [3]:
# create L2LState - the full episode state
# note: we use a mock data object here since we're just demonstrating the state model
class MockData:
    def list_drugs(self): return ["Trametinib", "Cobimetinib"]
    def list_cell_lines(self): return ["A549", "HCT116"]

with tempfile.TemporaryDirectory() as tmpdir:
    state = L2LState(
        episode_id="demo_001",
        episode_dir=Path(tmpdir),
        question_text="Why do MEK inhibitors suppress MAPK signaling?",
        target_pathway="MAPK signaling",
        budget=10,
        data=MockData(),
    )
    
    print("L2LState created:")
    print(f"  Episode ID: {state.episode_id}")
    print(f"  Question: {state.question_text}")
    print(f"  Target pathway: {state.target_pathway}")
    print(f"  Budget: {state.budget}")
    print(f"  Experiments run: {state.experiments_run}")
    print(f"  Done (aviary convention): {state.done}")
    print(f"  Reward (aviary convention): {state.reward}")

L2LState created:
  Episode ID: demo_001
  Question: Why do MEK inhibitors suppress MAPK signaling?
  Target pathway: MAPK signaling
  Budget: 10
  Experiments run: 0
  Done (aviary convention): False
  Reward (aviary convention): 0.0


In [4]:
# demonstrate aviary convention: tools can set state.done and state.reward directly
with tempfile.TemporaryDirectory() as tmpdir:
    state = L2LState(
        episode_id="demo_002",
        episode_dir=Path(tmpdir),
        question_text="Test question",
        target_pathway="Test pathway",
        budget=5,
        data=MockData(),
    )
    
    print("Before tool execution:")
    print(f"  done={state.done}, reward={state.reward}")
    
    # simulate what update_learned_knowledge(done=True) does
    state.done = True
    state.reward = 1.0
    
    print("\nAfter tool sets done=True:")
    print(f"  done={state.done}, reward={state.reward}")

Before tool execution:
  done=False, reward=0.0

After tool sets done=True:
  done=True, reward=1.0


## 2. TaskDataset Demo

The `L2LTaskDataset` wraps Question objects for standard aviary evaluation workflows.

In [5]:
# create mock questions for demonstration
class MockQuestion:
    def __init__(self, qid, question, pathway):
        self.id = qid
        self.question = question
        self.target_pathway = pathway
        self.test_set = []

questions = [
    MockQuestion("q_001", "Why do MEK inhibitors suppress MAPK signaling?", "MAPK signaling"),
    MockQuestion("q_002", "How do PI3K inhibitors affect AKT pathway?", "PI3K/AKT signaling"),
    MockQuestion("q_003", "What pathways are affected by EGFR inhibitors?", "EGFR signaling"),
    MockQuestion("q_004", "Why do BRAF inhibitors cause paradoxical activation?", "RAF/MEK/ERK cascade"),
    MockQuestion("q_005", "How do CDK4/6 inhibitors affect cell cycle?", "Cell cycle regulation"),
]

# create dataset
dataset = L2LTaskDataset(questions=questions, split="test", name="demo_dataset")

print(f"Dataset created: {dataset.name}")
print(f"  Length: {len(dataset)}")
print(f"  Split: {dataset.split}")

Dataset created: demo_dataset
  Length: 5
  Split: test


In [6]:
# iterate over dataset
print("Tasks in dataset:")
for i, task in enumerate(dataset):
    print(f"  {i+1}. [{task['question_id']}] {task['question_text'][:50]}...")

Tasks in dataset:
  1. [q_001] Why do MEK inhibitors suppress MAPK signaling?...
  2. [q_002] How do PI3K inhibitors affect AKT pathway?...
  3. [q_003] What pathways are affected by EGFR inhibitors?...
  4. [q_004] Why do BRAF inhibitors cause paradoxical activatio...
  5. [q_005] How do CDK4/6 inhibitors affect cell cycle?...


In [7]:
# index into dataset
task = dataset[2]
print("Task at index 2:")
print(f"  question_id: {task['question_id']}")
print(f"  question_text: {task['question_text']}")
print(f"  target_pathway: {task['target_pathway']}")
print(f"  split: {task['split']}")
print(f"  question object: {type(task['question']).__name__}")

Task at index 2:
  question_id: q_003
  question_text: What pathways are affected by EGFR inhibitors?
  target_pathway: EGFR signaling
  split: test
  question object: MockQuestion


In [8]:
# split dataset into train/val/test
train, val, test = dataset.get_train_val_test_split(
    train_frac=0.6,
    val_frac=0.2,
    seed=42
)

print("Dataset splits:")
print(f"  Train: {len(train)} questions (split='{train.split}')")
print(f"  Val: {len(val)} questions (split='{val.split}')")
print(f"  Test: {len(test)} questions (split='{test.split}')")
print(f"  Total: {len(train) + len(val) + len(test)}")

Dataset splits:
  Train: 3 questions (split='train')
  Val: 1 questions (split='val')
  Test: 1 questions (split='test')
  Total: 5


## 3. Scorer Demo

Scorers evaluate the quality of hypotheses and interpretations (S_process scoring).

In [9]:
import asyncio

# DummyScorer for testing - returns constant scores
scorer = DummyScorer(default_score=0.75)

async def demo_scoring():
    # score a hypothesis
    hyp_score, hyp_feedback = await scorer.score_hypothesis(
        hypothesis="MEK inhibitors will reduce MAPK pathway activity by blocking MEK1/2",
        question_context="Why do MEK inhibitors affect cancer cell proliferation?",
        available_experiments="All drug/cell line combinations"
    )
    print(f"Hypothesis score: {hyp_score}")
    print(f"Feedback: {hyp_feedback}")
    
    # score an interpretation
    interp_score, interp_feedback = await scorer.score_interpretation(
        interpretation="The observed negative NES confirms MEK inhibition reduces MAPK signaling",
        hypothesis="MEK inhibitors will reduce MAPK pathway activity",
        observation="MAPK pathway NES=-2.5, FDR=0.001"
    )
    print(f"\nInterpretation score: {interp_score}")
    print(f"Feedback: {interp_feedback}")

await demo_scoring()

Hypothesis score: 0.75
Feedback: Dummy score

Interpretation score: 0.75
Feedback: Dummy score


In [10]:
# show LLMScorer prompts (without actually calling the API)
print("LLMScorer hypothesis prompt template:")
print("-" * 50)
print(LLMScorer.HYPOTHESIS_PROMPT[:500] + "...")

LLMScorer hypothesis prompt template:
--------------------------------------------------
You are evaluating the quality of a scientific hypothesis.

Context: {question_context}

Available experiments: {available_experiments}

Hypothesis submitted:
{hypothesis}

Rate this hypothesis on a scale of 0.0 to 1.0 based on:
1. Testable: Can it be evaluated with available experiments?
2. Falsifiable: Is it clear what would disprove it?
3. Discriminating: Does it distinguish between competing explanations?

Respond with JSON:
{{"score": <float 0-1>, "feedback": "<brief explanation>"}}
...


## 4. Environment Demo

The `L2LEnvironment` is the main class that ties everything together. It provides a gymnasium-style interface for scientific reasoning episodes.

In [11]:
# create a more complete mock data object
class MockL2LData:
    """Mock L2LData for demonstration."""
    
    def list_drugs(self):
        return ["Trametinib", "Cobimetinib", "Binimetinib", "Selumetinib", "Gefitinib"]
    
    def list_cell_lines(self):
        return ["A549", "HCT116", "MCF7", "PC9", "H1975"]
    
    def get_drug(self, name):
        class MockDrug:
            def __init__(self, n):
                self.name = n
                self.moa_broad = "inhibitor/antagonist"
                self.moa_fine = "MEK inhibitor" if "metinib" in n.lower() else "EGFR inhibitor"
                self.targets = ["MEK1", "MEK2"] if "metinib" in n.lower() else ["EGFR"]
                self.human_approved = True
                self.clinical_trials = True
                self.pubchem_cid = 12345
        return MockDrug(name)
    
    def get_cell_line(self, name):
        class MockMutation:
            def __init__(self, gene):
                self.gene_symbol = gene
                self.protein_effect = "p.G12V" if gene == "KRAS" else "p.V600E"
                self.var_type = "Missense"
                self.mechanism = "GoF"
                self.gene_type = "Oncogene"
        
        class MockCellLine:
            def __init__(self, n):
                self.name = n
                self.organ = "lung" if n in ["A549", "PC9", "H1975"] else "colon"
                self.driver_mutations = [MockMutation("KRAS")] if n == "A549" else []
        return MockCellLine(name)
    
    def get_drugs_by_moa(self, moa, moa_type="fine"):
        if "MEK" in moa.upper():
            return ["Trametinib", "Cobimetinib", "Binimetinib"]
        return []
    
    def get_cell_lines_by_mutation(self, gene):
        import pandas as pd
        if gene == "KRAS":
            return pd.DataFrame({
                "cell_name": ["A549", "HCT116"],
                "Driver_ProtEffect_or_CdnaEffect": ["p.G12S", "p.G13D"]
            })
        return pd.DataFrame()
    
    def get_treatment(self, drug, concentration, cell_line):
        import pandas as pd
        
        class MockTreatment:
            def __init__(self, d, c, cl, data):
                self.drug = data.get_drug(d)
                self.concentration = c
                self.cell_line = data.get_cell_line(cl)
            
            def get_significant_pathways(self, fdr_threshold=0.05):
                return pd.DataFrame({
                    "pathway": ["MAPK signaling", "Cell cycle", "Apoptosis"],
                    "nes": [-2.5, -1.8, 1.2],
                    "fdr": [0.001, 0.01, 0.03]
                })
            
            def get_top_activated(self, n=5):
                return pd.DataFrame({
                    "pathway": ["Apoptosis", "p53 signaling"],
                    "nes": [1.2, 0.9],
                    "fdr": [0.03, 0.04]
                })
            
            def get_top_repressed(self, n=5):
                return pd.DataFrame({
                    "pathway": ["MAPK signaling", "Cell cycle", "DNA replication"],
                    "nes": [-2.5, -1.8, -1.5],
                    "fdr": [0.001, 0.01, 0.02]
                })
        
        return MockTreatment(drug, concentration, cell_line, self)

mock_data = MockL2LData()
print("MockL2LData created")
print(f"  Drugs: {mock_data.list_drugs()}")
print(f"  Cell lines: {mock_data.list_cell_lines()}")

MockL2LData created
  Drugs: ['Trametinib', 'Cobimetinib', 'Binimetinib', 'Selumetinib', 'Gefitinib']
  Cell lines: ['A549', 'HCT116', 'MCF7', 'PC9', 'H1975']


In [12]:
# create a mock question
class MockTestItem:
    def __init__(self, drug, conc, cell_line):
        class MockTreatment:
            def __init__(self, d, c, cl):
                self.drug = type('Drug', (), {'name': d})()
                self.concentration = c
                self.cell_line = type('CellLine', (), {'name': cl})()
        self.treatment = MockTreatment(drug, conc, cell_line)

class MockQuestionFull:
    def __init__(self):
        self.id = "demo_q_001"
        self.question = "Why do MEK inhibitors suppress MAPK signaling in KRAS-mutant cells?"
        self.target_pathway = "MAPK signaling"
        self.drug = "Trametinib"
        self.concentration = 0.05
        # held-out test set (these treatments cannot be queried)
        self.test_set = [
            MockTestItem("Trametinib", 0.05, "MCF7"),
            MockTestItem("Trametinib", 0.05, "PC9"),
        ]

mock_question = MockQuestionFull()
print(f"Question: {mock_question.question}")
print(f"Target pathway: {mock_question.target_pathway}")
print(f"Held-out test set size: {len(mock_question.test_set)}")

Question: Why do MEK inhibitors suppress MAPK signaling in KRAS-mutant cells?
Target pathway: MAPK signaling
Held-out test set size: 2


In [13]:
# create the environment
with tempfile.TemporaryDirectory() as tmpdir:
    env = L2LEnvironment(
        question=mock_question,
        data=mock_data,
        budget=5,
        scorer=DummyScorer(default_score=0.8),
        episodes_dir=Path(tmpdir),
    )
    
    print("L2LEnvironment created:")
    print(f"  Budget: {env.budget}")
    print(f"  Scorer: {type(env.scorer).__name__}")
    print(f"  Episodes dir: {env.episodes_dir}")

L2LEnvironment created:
  Budget: 5
  Scorer: DummyScorer
  Episodes dir: /var/folders/z_/fkp32h052nb7sr17w1jn8txw0000gn/T/tmp5pv46kfl


In [14]:
# reset the environment and examine initial state
async def demo_reset():
    with tempfile.TemporaryDirectory() as tmpdir:
        env = L2LEnvironment(
            question=mock_question,
            data=mock_data,
            budget=5,
            scorer=DummyScorer(),
            episodes_dir=Path(tmpdir),
        )
        
        # reset returns (messages, tools)
        messages, tools = await env.reset()
        
        print("Environment reset complete!")
        print(f"\nInitial message (context):")
        print("-" * 50)
        print(messages[0].content[:1000] + "...")
        print("-" * 50)
        
        print(f"\nAvailable tools ({len(tools)}):")
        for tool in tools:
            print(f"  - {tool.info.name}")
        
        print(f"\nState after reset:")
        print(f"  episode_id: {env.state.episode_id}")
        print(f"  experiments_run: {env.state.experiments_run}")
        print(f"  done: {env.state.done}")
        print(f"  reward: {env.state.reward}")
        print(f"  restricted_treatments: {env.state.restricted_treatments}")
        
        return env

env = await demo_reset()

Environment reset complete!

Initial message (context):
--------------------------------------------------
# L2L Benchmark Episode

## Your Task
Why do MEK inhibitors suppress MAPK signaling in KRAS-mutant cells?

## Target Pathway
MAPK signaling

## Experimental Budget
You have 5 experiments available. Used: 0/5

## Available Resources
- **Drugs**: 5 total (sample: Trametinib, Cobimetinib, Binimetinib, Selumetinib, Gefitinib...)
- **Cell lines**: 5 total (sample: A549, HCT116, MCF7, PC9, H1975...)

## Learned Knowledge

(No learned knowledge yet. Update after your first experiment.)

## Tools Available
Use the metadata tools to explore available drugs and cell lines.
Use submit_hypothesis() to submit your hypothesis and prediction before experiments.
Use run_experiment() to observe pathway activities (consumes budget).
Use submit_interpretation() after observing results.
Use update_learned_knowledge() to record insights and optionally end the episode.

## Scientific Process
For each e

In [15]:
# demonstrate tool usage - list drugs
async def demo_tools():
    with tempfile.TemporaryDirectory() as tmpdir:
        env = L2LEnvironment(
            question=mock_question,
            data=mock_data,
            budget=5,
            scorer=DummyScorer(),
            episodes_dir=Path(tmpdir),
        )
        await env.reset()
        
        print("=== Tool: list_available_drugs ===")
        result = env.list_available_drugs(env.state)
        print(result)
        
        print("\n=== Tool: get_drug_info ===")
        result = env.get_drug_info("Trametinib", env.state)
        print(result)
        
        print("\n=== Tool: get_cell_line_info ===")
        result = env.get_cell_line_info("A549", env.state)
        print(result)
        
        print("\n=== Tool: get_drugs_by_mechanism ===")
        result = env.get_drugs_by_mechanism("MEK inhibitor", env.state)
        print(result)
        
        print("\n=== Tool: get_cell_lines_with_mutation ===")
        result = env.get_cell_lines_with_mutation("KRAS", env.state)
        print(result)

await demo_tools()

=== Tool: list_available_drugs ===
Available drugs (5 total):
- Trametinib
- Cobimetinib
- Binimetinib
- Selumetinib
- Gefitinib

=== Tool: get_drug_info ===
# Trametinib
- Mechanism (broad): inhibitor/antagonist
- Mechanism (fine): MEK inhibitor
- Targets: MEK1, MEK2
- Human approved: Yes
- Clinical trials: Yes
- PubChem CID: 12345

=== Tool: get_cell_line_info ===
# A549
- Organ/tissue: lung
- Driver mutations (1):
  - KRAS (p.G12V): Missense [GoF] (Oncogene)

=== Tool: get_drugs_by_mechanism ===
Drugs matching 'MEK inhibitor':
- Trametinib
- Cobimetinib
- Binimetinib

=== Tool: get_cell_lines_with_mutation ===
Cell lines with KRAS mutations:
- A549: p.G12S
- HCT116: p.G13D


## 5. Full Episode Demo

Let's run through a complete hypothesis → experiment → interpretation cycle.

In [16]:
async def run_full_episode():
    """Run through a complete episode demonstrating the scientific process."""
    
    with tempfile.TemporaryDirectory() as tmpdir:
        env = L2LEnvironment(
            question=mock_question,
            data=mock_data,
            budget=3,
            scorer=DummyScorer(default_score=0.85),
            episodes_dir=Path(tmpdir),
        )
        
        # reset
        messages, tools = await env.reset()
        print("Episode started!")
        print(f"Episode ID: {env.state.episode_id}")
        print(f"Budget: {env.state.budget} experiments")
        print("=" * 60)
        
        # STEP 1: Submit hypothesis
        print("\n### Step 1: Submit Hypothesis ###")
        result = env.submit_hypothesis(
            hypothesis="MEK inhibitors like Trametinib will suppress MAPK pathway activity by blocking MEK1/2 kinase activity",
            alternatives="MAPK pathway activity may not be significantly affected if downstream compensation occurs",
            prediction="Expect negative NES (enrichment score) for MAPK signaling pathway after Trametinib treatment",
            confidence="High - MEK is a central node in MAPK cascade and Trametinib is a validated MEK inhibitor",
            state=env.state,
        )
        print(result)
        print(f"\nState: step={env.state.current_step}, hypothesis_set={env.state.current_hypothesis is not None}")
        
        # STEP 2: Run experiment
        print("\n### Step 2: Run Experiment ###")
        result = env.run_experiment(
            drug="Trametinib",
            concentration=0.05,
            cell_line="A549",
            state=env.state,
        )
        print(result)
        print(f"\nState: experiments_run={env.state.experiments_run}, awaiting_interpretation={env.state.awaiting_interpretation}")
        
        # STEP 3: Submit interpretation
        print("\n### Step 3: Submit Interpretation ###")
        result = env.submit_interpretation(
            interpretation="The results confirm our hypothesis - MAPK signaling shows strong repression (NES=-2.5, FDR=0.001). The cell cycle pathway is also repressed, consistent with downstream effects of MAPK inhibition.",
            hypothesis_supported="H1 - The negative NES and highly significant FDR strongly support that MEK inhibition suppresses MAPK signaling",
            state=env.state,
        )
        print(result)
        
        # STEP 4: Update learned knowledge (continue experimenting)
        print("\n### Step 4: Update Learned Knowledge ###")
        result = env.update_learned_knowledge(
            new_knowledge="""# Learned Knowledge

## Key Finding
- MEK inhibitor Trametinib strongly suppresses MAPK signaling in A549 cells (KRAS-mutant)
- NES = -2.5, FDR = 0.001 (highly significant)

## Secondary Observations
- Cell cycle pathway also repressed (expected downstream effect)
- Apoptosis pathway slightly activated (potential therapeutic effect)

## Next Steps
- Test in additional KRAS-mutant cell lines to confirm generalizability
""",
            done=False,  # continue experimenting
            state=env.state,
        )
        print(result)
        print(f"\nState: done={env.state.done}, reward={env.state.reward}")
        
        # verify knowledge file was created
        knowledge_files = list(env.state.episode_dir.glob("knowledge_*.md"))
        print(f"Knowledge files created: {[f.name for f in knowledge_files]}")
        
        # STEP 5: End episode
        print("\n### Step 5: End Episode ###")
        result = env.update_learned_knowledge(
            new_knowledge="""# Final Learned Knowledge

## Conclusion
MEK inhibitors suppress MAPK signaling by blocking MEK1/2 kinase activity.
This effect is robust in KRAS-mutant cell lines.
""",
            done=True,  # end episode
            state=env.state,
        )
        print(result)
        print(f"\nFinal State: done={env.state.done}, reward={env.state.reward}")
        
        # show episode summary
        print("\n" + "=" * 60)
        print("EPISODE SUMMARY")
        print("=" * 60)
        print(f"Episode ID: {env.state.episode_id}")
        print(f"Steps completed: {env.state.current_step}")
        print(f"Experiments run: {env.state.experiments_run}/{env.state.budget}")
        print(f"Final reward: {env.state.reward}")
        print(f"Episode done: {env.state.done}")
        
        return env

env = await run_full_episode()

Episode started!
Episode ID: 2d503102
Budget: 3 experiments

### Step 1: Submit Hypothesis ###
Hypothesis and prediction recorded for step 1.

H1: MEK inhibitors like Trametinib will suppress MAPK pathway activity by blocking MEK1/2 kinase activity
H2: MAPK pathway activity may not be significantly affected if downstream compensation occurs

Prediction: Expect negative NES (enrichment score) for MAPK signaling pathway after Trametinib treatment
Confidence: High - MEK is a central node in MAPK cascade and Trametinib is a validated MEK inhibitor

Now run an experiment using run_experiment().

State: step=1, hypothesis_set=True

### Step 2: Run Experiment ###
# Experiment Results: Trametinib @ 0.05uM in A549
Budget: 1/3 experiments used

## Cell Line Context
- Organ: lung
- Driver mutations: KRAS

## Drug Context
- Mechanism: MEK inhibitor
- Targets: MEK1, MEK2

## Pathway Activities
Significant pathways (FDR < 0.05): 3

### Top Activated Pathways (positive NES):
- Apoptosis: NES=1.20, FD

## 6. Restricted Treatments Demo

The environment enforces that held-out test set treatments cannot be queried.

In [17]:
async def demo_restricted_treatments():
    with tempfile.TemporaryDirectory() as tmpdir:
        env = L2LEnvironment(
            question=mock_question,
            data=mock_data,
            budget=5,
            scorer=DummyScorer(),
            episodes_dir=Path(tmpdir),
        )
        await env.reset()
        
        print("Restricted treatments (held-out test set):")
        for t in env.state.restricted_treatments:
            print(f"  - {t[0]} @ {t[1]}uM in {t[2]}")
        
        # submit hypothesis first
        env.submit_hypothesis(
            hypothesis="Test",
            alternatives="Alt",
            prediction="Pred",
            confidence="low",
            state=env.state,
        )
        
        # try to query a restricted treatment
        print("\nAttempting to query restricted treatment (Trametinib @ 0.05uM in MCF7):")
        result = env.run_experiment(
            drug="Trametinib",
            concentration=0.05,
            cell_line="MCF7",  # this is in the test set!
            state=env.state,
        )
        print(result)
        
        # query an allowed treatment
        print("\nQuerying allowed treatment (Trametinib @ 0.05uM in A549):")
        result = env.run_experiment(
            drug="Trametinib",
            concentration=0.05,
            cell_line="A549",  # this is NOT in the test set
            state=env.state,
        )
        print("Success!" if "Error" not in result else result)

await demo_restricted_treatments()

Restricted treatments (held-out test set):
  - Trametinib @ 0.05uM in PC9
  - Trametinib @ 0.05uM in MCF7

Attempting to query restricted treatment (Trametinib @ 0.05uM in MCF7):
Error: This treatment (Trametinib @ 0.05uM in MCF7) is in the held-out test set and cannot be queried.

Querying allowed treatment (Trametinib @ 0.05uM in A549):
Success!


## 7. Budget Exhaustion Demo

When the experimental budget is exhausted, `state.done` is set to True (aviary convention).

In [18]:
async def demo_budget_exhaustion():
    with tempfile.TemporaryDirectory() as tmpdir:
        env = L2LEnvironment(
            question=mock_question,
            data=mock_data,
            budget=2,  # only 2 experiments allowed
            scorer=DummyScorer(),
            episodes_dir=Path(tmpdir),
        )
        await env.reset()
        
        print(f"Budget: {env.state.budget} experiments")
        print(f"Initial done state: {env.state.done}")
        
        # run experiments until budget exhausted
        for i in range(3):  # try to run 3, but only 2 allowed
            # submit hypothesis
            env.submit_hypothesis(
                hypothesis=f"Hypothesis {i+1}",
                alternatives="Alt",
                prediction="Pred",
                confidence="low",
                state=env.state,
            )
            
            result = env.run_experiment(
                drug="Trametinib",
                concentration=0.05,
                cell_line="A549",
                state=env.state,
            )
            
            print(f"\nExperiment {i+1}:")
            if "Error" in result:
                print(f"  Result: {result}")
            else:
                print(f"  Success! experiments_run={env.state.experiments_run}")
            print(f"  done={env.state.done}")
            
            # reset hypothesis for next iteration if not done
            if not env.state.done and "Error" not in result:
                env.submit_interpretation(
                    interpretation="Interpretation",
                    hypothesis_supported="H1",
                    state=env.state,
                )

await demo_budget_exhaustion()

Budget: 2 experiments
Initial done state: False

Experiment 1:
  Success! experiments_run=1
  done=False

Experiment 2:
  Success! experiments_run=2
  done=True

Experiment 3:
  Result: Error: Budget exhausted. You have run 2/2 experiments.
  done=True


## Summary

This notebook demonstrated the key components of the L2L Bench aviary integration:

1. **L2LState**: Pydantic state model with `done`/`reward` following aviary convention
2. **L2LTaskDataset**: Wraps Questions for standard aviary evaluation workflows
3. **Scorers**: LLM-as-judge scoring (DummyScorer for testing, LLMScorer for production)
4. **L2LEnvironment**: Main environment with tools defined as methods
   - Metadata tools: `list_available_drugs`, `get_drug_info`, etc.
   - Scientific process tools: `submit_hypothesis`, `run_experiment`, `submit_interpretation`, `update_learned_knowledge`
   - Automatic enforcement of held-out test set restrictions
   - Budget tracking and automatic termination

The environment follows aviary patterns:
- Tools are methods on the Environment class
- State is passed as the last argument (hidden from agent)
- Tools set `state.done` and `state.reward` directly